# Pegasus PowerBox development sandbox

Sandbox for testing and developing ASCOM code to operate a Pegasus Astro PowerBox Gen 2.

The PPB is an instance of the ASCOM [Switch](https://ascom-standards.org/help/html/T_ASCOM_DriverAccess_Switch.htm) class.

In [90]:
from win32com.client import Dispatch
from time import sleep

# time to pause for actions in seconds

twait = 1.0 # seconds

## Connect to the Pegasus

Device ID is `ASCOM.Device1.Switch`. Use the ASCOM Profile Explorer app to identify if in doubt.

`ppb.Connect=True` establishes the connection.

In [91]:
ppb = Dispatch("ASCOM.Device1.Switch")

ppb.Connected = True
print(f"Connected to {ppb.Name}, number of switches: {ppb.MaxSwitch}")

Connected to PPBAdvance PPBAAOFAKOA, number of switches: 9


## List the switches on the Pegasus

Looks at each device on the pegasus and gives the power state ON or OFF

Note that for this box, only 0..3 are switchable power outputs:
 * 0 = Quad 12VDC power output ports (all 4 on or off, not individual)
 * 1 = 0..12VDC adjustable power output port
 * 2 = DewA - Dew heater output A (0..100%)
 * 3 = DewB - Dew heater output B (0..100%)

The next three "switches" are readouts:
 * 4 = unit input voltage (VDC)
 * 5 = unit input current (amps)
 * 6 = unit input power (amps)

The last two switches are ON/Off function toggles:
 * 7 = AutoDew - automatic dew heater on/off (value is agressiveness?)
 * 8 = Dual USB2 Port - power off the dual USB2 ports if unused (default: ON)

In [92]:
for i in range(ppb.MaxSwitch):
    swState = ppb.GetSwitch(i)
    swDesc = ppb.GetSwitchDescription(i)
    swVal = ppb.GetSwitchValue(i)
    swName = ppb.GetSwitchName(i)
    swState = ppb.GetSwitch(i)
    swMin = ppb.MinSwitchValue(i)
    swMax = ppb.MaxSwitchValue(i)
    print(f"Switch {i}: {swDesc} - value: {swVal} range: {swMin}..{swMax}, {swName}, state: {'ON' if swState else 'OFF'}")
    

Switch 0: Quad Power - value: 1.0 range: 0.0..1.0, Quad Power, state: ON
Switch 1: Adjustable - value: 0.0 range: 0.0..1.0, Adjustable, state: OFF
Switch 2: Dew A - value: 0.0 range: 0.0..100.0, Dew A, state: OFF
Switch 3: Dew B - value: 0.0 range: 0.0..100.0, Dew B, state: OFF
Switch 4: Voltage - value: 12.6 range: 0.0..16.0, Input Voltage, state: ON
Switch 5: Current - value: 0.13 range: 0.0..30.0, Amp, state: ON
Switch 6: Power - value: 1.0 range: 0.0..30.0, Watt, state: ON
Switch 7: AutoDew - value: 0.0 range: 0.0..1.0, AutoDew, state: OFF
Switch 8: Dual USB2 Port - value: 1.0 range: 0.0..1.0, Dual USB2 Port, state: ON


## Toggle a switch on or off

The `SetSwitch()` action takes time and is asynchronous, so introduce a pause of `twait`

In [88]:
iSw = 1
swVal = "off"

ppb.SetSwitch(iSw,swVal.upper()=="ON")
sleep(twait)
print(f"Switch {iSw}: state: {'ON' if ppb.GetSwitch(iSw) else 'OFF'}")

Switch 3: state: OFF


## Set variable output port value


In [76]:
iSw = 2  # DewA heater value, range 0..100
swVal = 50.
ppb.SetSwitch(iSw,True)
ppb.SetSwitchValue(iSw,swVal)
sleep(twait)
print(f"Switch {iSw}: state: {'ON' if ppb.GetSwitch(iSw) else 'OFF'}, value: {ppb.GetSwitchValue(iSw)}")
sleep(2)
ppb.SetSwitchValue(iSw,30.0)
sleep(2*twait)
print(f"Switch {iSw}: state: {'ON' if ppb.GetSwitch(iSw) else 'OFF'}, value: {ppb.GetSwitchValue(iSw)}")
#sleep(2)
#ppb.SetSwitch(iSw,False)
#sleep(twait)
#print(f"Switch {iSw}: state: {'ON' if ppb.GetSwitch(iSw) else 'OFF'}, value: {ppb.GetSwitchValue(iSw)}")


Switch 2: state: ON, value: 50.0
Switch 2: state: ON, value: 30.0
